In [1]:
import sys, os, json, warnings
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import mlflow
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Optuna version:', optuna.__version__)

Optuna version: 4.9.0


In [2]:
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('04_Final_Tuning')

TREE_DIR    = 'data/processed/tree_models'
CB_DIR      = 'data/processed/catboost'
CHECKPOINTS = 'artifacts/checkpoints'
N_TRIALS    = 50   # reduce to 20 for faster run

# Load data
tree_train = pd.read_parquet(f'{TREE_DIR}/train.parquet')
tree_val   = pd.read_parquet(f'{TREE_DIR}/val.parquet')
cb_train   = pd.read_parquet(f'{CB_DIR}/train.parquet')
cb_val     = pd.read_parquet(f'{CB_DIR}/val.parquet')

# Load VAE scores (from exp 3 checkpoints)
try:
    tree_train['vae_anomaly_score'] = np.load(f'{CHECKPOINTS}/vae_train_scores.npy')
    tree_val['vae_anomaly_score']   = np.load(f'{CHECKPOINTS}/vae_val_scores.npy')
    cb_train['vae_anomaly_score']   = tree_train['vae_anomaly_score'].values
    cb_val['vae_anomaly_score']     = tree_val['vae_anomaly_score'].values
    print('VAE scores loaded.')
except FileNotFoundError:
    print('WARNING: VAE scores not found. Run Exp 3 first.')

EXCLUDE = ['TransactionID', 'isFraud', 'uid1', 'uid2', 'uid3', 'uid_magic',
           'card4', 'card6', 'ProductCD', 'DeviceType', 'P_emaildomain',
           'R_emaildomain', 'DeviceInfo', 'device_os', 'device_browser',
           'P_email_tld', 'R_email_tld']

feat_cols = [c for c in tree_train.columns
             if c not in EXCLUDE and tree_train[c].dtype != object]

X_train = tree_train[feat_cols].values
y_train = tree_train['isFraud'].values
X_val   = tree_val[feat_cols].values
y_val   = tree_val['isFraud'].values
print(f'Features: {len(feat_cols)}  Train: {len(X_train)}  Val: {len(X_val)}')

VAE scores loaded.
Features: 654  Train: 337352  Val: 107821


In [ ]:
# ── Run 4a: XGBoost Optuna HPO ────────────────────────────────────────────────
print(f'Running XGBoost Optuna ({N_TRIALS} trials)…')

def xgb_objective(trial):
    params = {
        'n_estimators':     3000,
        'max_depth':        trial.suggest_int('max_depth', 6, 14),
        'learning_rate':    trial.suggest_float('learning_rate', 0.001, 0.02, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.7),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 0.5, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 0.5, log=True),
        'scale_pos_weight': 27,
        'tree_method':      'hist',
        'objective':        'binary:logistic',
        'eval_metric':      'aucpr',
        'random_state':     42,
        'n_jobs':           -1,
    }
    # TO THIS:
    model = xgb.XGBClassifier(early_stopping_rounds=100, **params)
    model.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],
          verbose=False)

    preds = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, preds)

xgb_study = optuna.create_study(direction='maximize',
                                 study_name='xgb_ieee_cis',
                                 sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_xgb_params = xgb_study.best_params
best_xgb_aupr   = xgb_study.best_value
print(f'\nBest XGBoost AUC-PR: {best_xgb_aupr:.5f}')
print('Best params:', best_xgb_params)

In [ ]:
# Train final XGBoost with best params + 5000 trees
final_xgb_params = {**best_xgb_params,
    'n_estimators': 5000, 'scale_pos_weight': 27, 'tree_method': 'hist',
    'objective': 'binary:logistic', 'random_state': 42, 'n_jobs': -1,'device' : 'cuda','early_stopping_rounds':200 }

with mlflow.start_run(run_name='4a_xgb_optuna_best') as run:
    mlflow.log_params(final_xgb_params)
    mlflow.log_params({'n_optuna_trials': N_TRIALS, 'model_type': 'XGBoost'})

    xgb_final = xgb.XGBClassifier(**final_xgb_params)
    xgb_final.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                   verbose=False)

    xgb_val_preds = xgb_final.predict_proba(X_val)[:, 1]
    xgb_final_auc  = roc_auc_score(y_val, xgb_val_preds)
    xgb_final_aupr = average_precision_score(y_val, xgb_val_preds)

    mlflow.log_metrics({'val_auc_roc': xgb_final_auc, 'val_auc_pr': xgb_final_aupr,
                        'optuna_best_aupr': best_xgb_aupr})

    # Log Optuna plot
    try:
        fig = optuna.visualization.plot_optimization_history(xgb_study)
        fig.write_image('artifacts/shap/xgb_optuna_history.png')
        mlflow.log_artifact('artifacts/shap/xgb_optuna_history.png')
    except: pass

    print(f'Final XGBoost — AUC-ROC: {xgb_final_auc:.5f}  AUC-PR: {xgb_final_aupr:.5f}')

In [ ]:
# ── Run 4b: LightGBM Optuna HPO ───────────────────────────────────────────────
print(f'Running LightGBM Optuna ({N_TRIALS} trials)…')

def lgbm_objective(trial):
    params = {
        'n_estimators':      3000,
        'num_leaves':        trial.suggest_int('num_leaves', 64, 1024),
        'learning_rate':     trial.suggest_float('learning_rate', 0.001, 0.02, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.3, 0.7),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 0.5, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 0.5, log=True),
        'scale_pos_weight':  27,
        'objective':         'binary',
        'random_state':      42,
        'verbose':          -1,
        'n_jobs':           -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(9999)])
    preds = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, preds)

lgbm_study = optuna.create_study(direction='maximize', study_name='lgbm_ieee_cis',
                                  sampler=optuna.samplers.TPESampler(seed=42))
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_lgbm_params = lgbm_study.best_params
print(f'Best LightGBM AUC-PR: {lgbm_study.best_value:.5f}')

# Train final
final_lgbm_params = {**best_lgbm_params, 'n_estimators': 5000,
    'scale_pos_weight': 27, 'objective': 'binary', 'random_state': 42,
    'verbose': -1, 'n_jobs': -1}

with mlflow.start_run(run_name='4b_lgbm_optuna_best') as run:
    mlflow.log_params(final_lgbm_params)
    lgbm_final = lgb.LGBMClassifier(**final_lgbm_params)
    lgbm_final.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                   callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(9999)])
    lgbm_val_preds = lgbm_final.predict_proba(X_val)[:, 1]
    lgbm_final_auc  = roc_auc_score(y_val, lgbm_val_preds)
    lgbm_final_aupr = average_precision_score(y_val, lgbm_val_preds)
    mlflow.log_metrics({'val_auc_roc': lgbm_final_auc, 'val_auc_pr': lgbm_final_aupr})
    print(f'Final LightGBM — AUC-ROC: {lgbm_final_auc:.5f}  AUC-PR: {lgbm_final_aupr:.5f}')

In [ ]:
# ── Run 4c: CatBoost Optuna HPO ───────────────────────────────────────────────
CAT_COLS = ['card4','card6','ProductCD','DeviceType','P_emaildomain']
cb_feat_cols = [c for c in cb_train.columns
                if c not in ['TransactionID','isFraud'] and
                (cb_train[c].dtype != object or c in CAT_COLS)]
cat_idx = [cb_feat_cols.index(c) for c in CAT_COLS if c in cb_feat_cols]

def catboost_objective(trial):
    params = {
        'iterations':           1500,
        'depth':                trial.suggest_int('depth', 5, 10),
        'learning_rate':        trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'l2_leaf_reg':          trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature':  trial.suggest_float('bagging_temperature', 0.0, 2.0),
        'scale_pos_weight':     27,
        'eval_metric':          'PRAUC',
        'loss_function':        'Logloss',
        'random_seed':          42,
        'verbose':              0,
        'early_stopping_rounds':100,
    }
    tr_pool = Pool(cb_train[cb_feat_cols], cb_train['isFraud'], cat_features=cat_idx)
    vl_pool = Pool(cb_val[cb_feat_cols],   cb_val['isFraud'],   cat_features=cat_idx)
    model = CatBoostClassifier(**params)
    model.fit(tr_pool, eval_set=vl_pool, use_best_model=True)
    preds = model.predict_proba(vl_pool)[:, 1]
    return average_precision_score(cb_val['isFraud'], preds)

print('Running CatBoost Optuna (30 trials)…')
cb_study = optuna.create_study(direction='maximize', study_name='catboost_ieee_cis')
cb_study.optimize(catboost_objective, n_trials=30, show_progress_bar=True)

best_cb_params = {**cb_study.best_params,
    'iterations': 5000, 'scale_pos_weight': 27, 'eval_metric': 'PRAUC',
    'loss_function': 'Logloss', 'random_seed': 42, 'verbose': 500,
    'early_stopping_rounds': 200}

with mlflow.start_run(run_name='4c_catboost_optuna_best') as run:
    mlflow.log_params(best_cb_params)
    tr_pool = Pool(cb_train[cb_feat_cols], cb_train['isFraud'], cat_features=cat_idx)
    vl_pool = Pool(cb_val[cb_feat_cols],   cb_val['isFraud'],   cat_features=cat_idx)
    cb_final = CatBoostClassifier(**best_cb_params)
    cb_final.fit(tr_pool, eval_set=vl_pool, use_best_model=True)
    cb_val_preds = cb_final.predict_proba(vl_pool)[:, 1]
    cb_final_auc  = roc_auc_score(cb_val['isFraud'], cb_val_preds)
    cb_final_aupr = average_precision_score(cb_val['isFraud'], cb_val_preds)
    mlflow.log_metrics({'val_auc_roc': cb_final_auc, 'val_auc_pr': cb_final_aupr})
    print(f'Final CatBoost — AUC-ROC: {cb_final_auc:.5f}  AUC-PR: {cb_final_aupr:.5f}')

In [ ]:
# ── Run 4d: Full 7-model stacking (LogReg meta-learner) ───────────────────────
from src.models.stacking.meta_learner import (assemble_meta_features,
    apply_decision_thresholds, STACKING_INPUT_COLS)

# Load DL probs from checkpoints
try:
    bilstm_val_probs = np.load(f'{CHECKPOINTS}/bilstm_val_probs.npy')
    gnn_val_probs    = np.load(f'{CHECKPOINTS}/gnn_val_probs.npy')
    gnn_ring_risk    = np.load(f'{CHECKPOINTS}/gnn_ring_risk.npy')
    vae_val_scores   = np.load(f'{CHECKPOINTS}/vae_val_scores.npy')
    vae_train_scores = np.load(f'{CHECKPOINTS}/vae_train_scores.npy')
    print('DL checkpoints loaded.')
except FileNotFoundError as e:
    print(f'Missing checkpoint: {e} — run Exp 3 first')

n_val = len(y_val)
val_probs = {
    'xgb_prob':          xgb_val_preds[:n_val],
    'lgbm_prob':         lgbm_val_preds[:n_val],
    'catboost_prob':     cb_val_preds[:n_val],
    'vae_anomaly_score': vae_val_scores[:n_val],
    'bilstm_prob':       bilstm_val_probs[:n_val],
    'gnn_fraud_prob':    gnn_val_probs[:n_val],
    'ring_risk_score':   gnn_ring_risk[:n_val],
}

# Use XGB OOF preds for train meta-features
xgb_oof  = np.load(f'{CHECKPOINTS}/xgb_oof_preds.npy') if os.path.exists(f'{CHECKPOINTS}/xgb_oof_preds.npy') else xgb_val_preds
lgbm_oof = np.load(f'{CHECKPOINTS}/lgbm_oof_preds.npy') if os.path.exists(f'{CHECKPOINTS}/lgbm_oof_preds.npy') else lgbm_val_preds
cb_oof   = np.load(f'{CHECKPOINTS}/catboost_oof_preds.npy') if os.path.exists(f'{CHECKPOINTS}/catboost_oof_preds.npy') else cb_val_preds

n_train = len(y_train)
train_probs = {
    'xgb_prob':          xgb_oof[:n_train],
    'lgbm_prob':         lgbm_oof[:n_train],
    'catboost_prob':     cb_oof[:n_train],
    'vae_anomaly_score': vae_train_scores[:n_train],
    'bilstm_prob':       np.full(n_train, bilstm_val_probs.mean()),  # approx
    'gnn_fraud_prob':    np.full(n_train, gnn_val_probs.mean()),
    'ring_risk_score':   np.full(n_train, gnn_ring_risk.mean()),
}

# Build meta-feature matrices
X_meta_train = assemble_meta_features(train_probs)
X_meta_val   = assemble_meta_features(val_probs)

# Scale + train calibrated LogReg
scaler = StandardScaler()
X_meta_train_s = scaler.fit_transform(X_meta_train)
X_meta_val_s   = scaler.transform(X_meta_val)

base_lr = LogisticRegression(C=1.0, max_iter=1000, class_weight={0:1, 1:27}, random_state=42)
meta_model = CalibratedClassifierCV(base_lr, method='isotonic', cv=5)
meta_model.fit(X_meta_train_s, y_train[:n_train])

final_preds = meta_model.predict_proba(X_meta_val_s)[:, 1]
final_auc   = roc_auc_score(y_val, final_preds)
final_aupr  = average_precision_score(y_val, final_preds)

with mlflow.start_run(run_name='4d_full_stack_7models') as run:
    mlflow.log_params({'meta_model': 'CalibratedLogReg', 'n_base_models': 7,
                       'calibration': 'isotonic', 'cv_folds': 5})
    mlflow.log_metrics({'val_auc_roc': final_auc, 'val_auc_pr': final_aupr})
    for k, arr in val_probs.items():
        mlflow.log_metric(f'base_{k}_aupr', average_precision_score(y_val, arr))

print(f'\nFull Stack — AUC-ROC: {final_auc:.5f}  AUC-PR: {final_aupr:.5f}')

In [ ]:
# ── Run 4e: Optuna stacking weight search ─────────────────────────────────────
model_keys = list(val_probs.keys())
val_arrays = [val_probs[k] for k in model_keys]

def weight_objective(trial):
    weights = np.array([trial.suggest_float(k, 0.0, 1.0) for k in model_keys])
    weights /= weights.sum()  # normalise to sum=1
    combined = sum(w * arr for w, arr in zip(weights, val_arrays))
    return average_precision_score(y_val, combined)

print('Running stacking weight search (30 trials)…')
weight_study = optuna.create_study(direction='maximize')
weight_study.optimize(weight_objective, n_trials=30, show_progress_bar=True)

best_weights = {k: weight_study.best_params[k] for k in model_keys}
w_arr = np.array([best_weights[k] for k in model_keys])
w_arr /= w_arr.sum()
weighted_preds = sum(w * arr for w, arr in zip(w_arr, val_arrays))
weighted_aupr  = average_precision_score(y_val, weighted_preds)

with mlflow.start_run(run_name='4e_optuna_weights') as run:
    mlflow.log_params({f'w_{k}': round(w_arr[i], 4) for i, k in enumerate(model_keys)})
    mlflow.log_metrics({'val_auc_pr': weighted_aupr,
                        'val_auc_roc': roc_auc_score(y_val, weighted_preds)})

print(f'Optimal weights: {dict(zip(model_keys, w_arr.round(3)))}')
print(f'Weighted AUC-PR: {weighted_aupr:.5f}')

In [ ]:
# ── Final summary + decision distribution ─────────────────────────────────────
best_preds = weighted_preds if weighted_aupr > final_aupr else final_preds
best_aupr  = max(weighted_aupr, final_aupr)

decisions = apply_decision_thresholds(best_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR curve
from sklearn.metrics import precision_recall_curve
prec, rec, _ = precision_recall_curve(y_val, best_preds)
axes[0].plot(rec, prec, color='#2ecc71', lw=2)
axes[0].fill_between(rec, prec, alpha=0.1, color='#2ecc71')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title(f'Final Stack PR Curve (AUC-PR={best_aupr:.4f})', fontsize=13)
axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])

# Decision distribution
dec_counts = decisions['decision'].value_counts()
colors = {'auto_approve': '#2ecc71', 'flag_review': '#f39c12', 'block': '#e74c3c'}
bars = axes[1].bar(dec_counts.index,
                   dec_counts.values / len(decisions),
                   color=[colors.get(k,'gray') for k in dec_counts.index])
for bar, val in zip(bars, dec_counts.values/len(decisions)):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('Decision Distribution', fontsize=13)
axes[1].set_ylabel('% of transactions')
axes[1].set_ylim([0,1])

plt.tight_layout()
plt.savefig('artifacts/shap/exp04_final_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Load all experiment metrics and print comparison ─────────────────────────
all_exp = {}
for fn in ['exp01_baseline.json', 'exp02_feature_engineering.json',
           'exp03_deep_learning.json']:
    try:
        with open(f'artifacts/metrics/{fn}') as f:
            d = json.load(f)
            all_exp[d['experiment']] = d.get('val_auc_pr', d.get('best_aupr', 0))
    except: pass

all_exp['04_final_tuning'] = best_aupr

print('\n' + '='*55)
print(f'{"EXPERIMENT":35s}  {"AUC-PR":>8s}  {"Δ Baseline":>10s}')
print('='*55)
baseline = all_exp.get('01_baseline', 0)
for exp, aupr in all_exp.items():
    delta = f'+{aupr-baseline:.5f}' if exp != '01_baseline' else '---'
    print(f'{exp:35s}  {aupr:8.5f}  {delta:>10s}')
print('='*55)

# Save final metrics
final_metrics = {
    'experiment': '04_final_tuning',
    'val_auc_pr':  round(best_aupr, 5),
    'val_auc_roc': round(roc_auc_score(y_val, best_preds), 5),
    'decision_distribution': decisions['decision'].value_counts().to_dict(),
    'best_xgb_params':  best_xgb_params,
    'best_lgbm_params': best_lgbm_params,
    'best_cb_params':   cb_study.best_params,
    'optimal_weights':  dict(zip(model_keys, w_arr.round(4).tolist())),
}
with open('artifacts/metrics/exp04_final_tuning.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

with mlflow.start_run(run_name='4_FINAL_BEST'):
    mlflow.log_artifact('artifacts/shap/exp04_final_results.png')
    mlflow.log_artifact('artifacts/metrics/exp04_final_tuning.json')
    mlflow.log_metrics({'final_val_auc_pr': best_aupr,
                        'total_improvement': best_aupr - baseline})

print('\nAll experiments complete. Open MLflow UI: http://localhost:5000')
print('Run: mlflow server --backend-store-uri sqlite:///mlflow_tracking/mlflow.db --port 5000')